In [2]:
import pandas as pd
import numpy as np

# ==========================================================
# PACKAGE 7
# FORECAST DATASET + LEAKAGE CONTROL
# ==========================================================

INPUT_FILE = r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Data_Integration\Delhi_Master_Dataset_Final.csv"
OUTPUT_FILE = r"Delhi_Forecasting_Dataset.csv"

print("=" * 60)
print("PACKAGE 7 : FORECAST DATASET PREPARATION")
print("=" * 60)

# ==========================================================
# LOAD
# ==========================================================

df = pd.read_csv(INPUT_FILE)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print(f"\nOriginal Shape : {df.shape}")

# ==========================================================
# CREATE NEXT-DAY TARGET
# ==========================================================

df["AQI_Target"] = df["AQI"].shift(-1)

# remove last row (no target)
df = df.iloc[:-1].copy()

print("\nTarget Created : AQI_Target (Next-Day AQI)")

# ==========================================================
# REMOVE TRUE LEAKAGE
# ==========================================================

remove_columns = [
    "AQI_Bucket",
    "AQI_Bucket_Encoded",
    "AQI_Diff"
]

removed = []

for col in remove_columns:
    if col in df.columns:
        df.drop(columns=col, inplace=True)
        removed.append(col)

print("\nLeakage Features Removed:")
for c in removed:
    print(" -", c)

# ==========================================================
# REMOVE DUPLICATE COLUMNS
# ==========================================================

before = df.shape[1]

df = df.loc[:, ~df.columns.duplicated()]

after = df.shape[1]

print(f"\nDuplicate Columns Removed : {before-after}")

# ==========================================================
# REMOVE CONSTANT FEATURES
# ==========================================================

constant_columns = []

for col in df.columns:

    if col in ["Date", "City", "AQI", "AQI_Target"]:
        continue

    if df[col].nunique(dropna=False) <= 1:
        constant_columns.append(col)

df.drop(columns=constant_columns, inplace=True)

print(f"\nConstant Features Removed : {len(constant_columns)}")

# ==========================================================
# REMOVE ALL-NaN COLUMNS
# ==========================================================

all_nan = df.columns[df.isna().all()].tolist()

if len(all_nan) > 0:
    df.drop(columns=all_nan, inplace=True)

print(f"All-NaN Columns Removed : {len(all_nan)}")

# ==========================================================
# MISSING VALUE REPORT
# ==========================================================

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("\nTop Missing Values:")
print(missing.head(20))

# ==========================================================
# SAVE FEATURE REMOVAL REPORT
# ==========================================================

report = pd.DataFrame({
    "Removed_Feature":
        removed
        + constant_columns
        + all_nan,

    "Reason":
        ["Leakage"] * len(removed)
        + ["Constant"] * len(constant_columns)
        + ["All_NaN"] * len(all_nan)
})

report.to_csv(
    "Feature_Removal_Report.csv",
    index=False
)

# ==========================================================
# SAVE DATASET
# ==========================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("FINAL DATASET CREATED")
print("=" * 60)

print("Shape :", df.shape)

print("\nTarget:")
print("AQI_Target")

print("\nDate Range:")
print(df["Date"].min(), "to", df["Date"].max())

print("\nSaved Files:")
print("1. Delhi_Forecasting_Dataset.csv")
print("2. Feature_Removal_Report.csv")

print("\nPackage 7 Completed Successfully.")

PACKAGE 7 : FORECAST DATASET PREPARATION

Original Shape : (2009, 240)

Target Created : AQI_Target (Next-Day AQI)

Leakage Features Removed:
 - AQI_Bucket
 - AQI_Bucket_Encoded
 - AQI_Diff

Duplicate Columns Removed : 0

Constant Features Removed : 156
All-NaN Columns Removed : 0

Top Missing Values:
Sentinel_NO2_Lag3    1360
Sentinel_AAI_Lag3    1360
Sentinel_AAI_Lag1    1358
Sentinel_NO2_Lag1    1358
Sentinel_NO2_MA3     1357
Sentinel_AAI         1357
Sentinel_NO2         1357
Sentinel_AAI_MA3     1357
MODIS_AOD_Lag3       1117
MODIS_AOD_Lag1       1115
MODIS_AOD_MA7        1114
MODIS_AOD            1114
MODIS_AOD_MA3        1114
dtype: int64

FINAL DATASET CREATED
Shape : (2008, 82)

Target:
AQI_Target

Date Range:
2015-01-01 00:00:00 to 2020-06-30 00:00:00

Saved Files:
1. Delhi_Forecasting_Dataset.csv
2. Feature_Removal_Report.csv

Package 7 Completed Successfully.


In [4]:
# ============================================================
# SATELLITE AVAILABILITY SUMMARY REPORT
# PACKAGE 7 DOCUMENTATION ADD-ON
# ============================================================

import pandas as pd


print("="*70)
print("SATELLITE AVAILABILITY SUMMARY REPORT")
print("="*70)


# Load forecasting dataset

df = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecast_Dataset_Leakage_Removal\Delhi_Forecasting_Dataset.csv"
)


df["Date"] = pd.to_datetime(df["Date"])



# ------------------------------------------------------------
# Satellite Availability Columns
# ------------------------------------------------------------

satellite_flags = [
    "MODIS_available",
    "Sentinel_available"
]


summary = []



for col in satellite_flags:

    available = df[col].sum()

    unavailable = len(df) - available

    coverage = (
        available / len(df)
    ) * 100


    summary.append(
        {
            "Satellite":
                col.replace("_available",""),

            "Available_Days":
                int(available),

            "Unavailable_Days":
                int(unavailable),

            "Coverage_Percentage":
                round(coverage,2)
        }
    )



summary_df = pd.DataFrame(summary)



print("\n============================================================")
print("SATELLITE COVERAGE SUMMARY")
print("============================================================")


print(summary_df)



# ------------------------------------------------------------
# Actual Feature Availability
# ------------------------------------------------------------

satellite_features = [
    c for c in df.columns
    if (
        "MODIS" in c
        or
        "Sentinel" in c
    )
    and
    "available" not in c
]


feature_summary = pd.DataFrame(
    {
        "Feature":
            satellite_features,

        "Available_Count":
            [
                df[c].notna().sum()
                for c in satellite_features
            ],

        "Missing_Count":
            [
                df[c].isna().sum()
                for c in satellite_features
            ],

        "Coverage_Percentage":
            [
                round(
                    df[c].notna().mean()*100,
                    2
                )
                for c in satellite_features
            ]
    }
)



print("\n============================================================")
print("SATELLITE FEATURE COVERAGE")
print("============================================================")


print(feature_summary)



# ------------------------------------------------------------
# Save Reports
# ------------------------------------------------------------

summary_df.to_csv(
    "Satellite_Availability_Summary.csv",
    index=False
)


feature_summary.to_csv(
    "Satellite_Feature_Coverage_Report.csv",
    index=False
)



print("\nSaved Files:")
print("1. Satellite_Availability_Summary.csv")
print("2. Satellite_Feature_Coverage_Report.csv")


print("\nSatellite Availability Report Completed.")

SATELLITE AVAILABILITY SUMMARY REPORT

SATELLITE COVERAGE SUMMARY
  Satellite  Available_Days  Unavailable_Days  Coverage_Percentage
0     MODIS             894              1114                44.52
1  Sentinel             651              1357                32.42

SATELLITE FEATURE COVERAGE
              Feature  Available_Count  Missing_Count  Coverage_Percentage
0           MODIS_AOD              894           1114                44.52
1      MODIS_AOD_Lag1              893           1115                44.47
2      MODIS_AOD_Lag3              891           1117                44.37
3       MODIS_AOD_MA3              894           1114                44.52
4       MODIS_AOD_MA7              894           1114                44.52
5        Sentinel_NO2              651           1357                32.42
6   Sentinel_NO2_Lag1              650           1358                32.37
7   Sentinel_NO2_Lag3              648           1360                32.27
8    Sentinel_NO2_MA3         